In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!pip install yt-dlp

In [ ]:
import yt_dlp

youtube_url = input("Enter YouTube URL: ")

# -------------------------------
# Method 1 : Normal extraction
# -------------------------------
method1_opts = {
    'format': 'bestaudio/best',
    'outtmpl': 'audio.%(ext)s',
}

# -------------------------------
# Method 2 : Android fallback
# -------------------------------
method2_opts = {
    'format': 'bestaudio/best',
    'outtmpl': 'audio.%(ext)s',
    'extractor_args': {
        'youtube': {
            'player_client': ['android']
        }
    }
}

try:
    print("\nTrying Method 1 (Normal Extraction)...\n")

    with yt_dlp.YoutubeDL(method1_opts) as ydl:
        ydl.download([youtube_url])

    print("\nAudio downloaded successfully using Method 1!")

except Exception as e:

    print("\nMethod 1 failed!")
    print("Error:", e)

    print("\nTrying Method 2 (Android Fallback)...\n")

    try:
        with yt_dlp.YoutubeDL(method2_opts) as ydl:
            ydl.download([youtube_url])

        print("\nAudio downloaded successfully using Method 2!")

    except Exception as e2:

        print("\nMethod 2 also failed!")
        print("Error:", e2)

        print("\nCould not extract audio from this URL.")

format change code

In [ ]:
# =====================================================
# INSTALL DEPENDENCIES
# =====================================================
!pip install -q transformers accelerate huggingface_hub sentencepiece
!pip install -q torch torchaudio
!apt update -qq
!apt install -qq ffmpeg

In [ ]:
!ffmpeg -i audio.m4a -ar 16000 -ac 1 audio.wav -y

format change

In [ ]:
# Install required libraries
!pip install -q transformers accelerate huggingface_hub sentencepiece

# Install ffmpeg (required for audio processing)
!apt-get -qq install ffmpeg

In [ ]:
import shutil

shutil.move("/audio.wav", "/content/audio.wav")

print("Moved to /content ✔")

try new


In [ ]:
!echo "📦 Installing dependencies..."
!pip install -q transformers accelerate huggingface_hub sentencepiece
!pip install -q torch torchaudio
!pip install -q pydub
!apt update -qq
!apt install -qq ffmpeg

!echo "✅ Installation completed"

In [ ]:
!echo "📥 Loading Python libraries..."

import os
import torch
from transformers import pipeline
from huggingface_hub import snapshot_download
from google.colab import drive
from pydub import AudioSegment

!echo "✅ Libraries loaded"

In [ ]:
!echo "🔗 Mounting Google Drive..."

drive.mount('/content/drive')

!echo "✅ Drive mounted"

In [ ]:
!echo "⚙️ Setting configuration..."

MODEL_ID = "openai/whisper-large-v3"
LOCAL_DIR = "/content/drive/MyDrive/ai_models/whisper_large_v3"
AUDIO_FILE = "/content/audio.wav"

!echo "Model ID set to: openai/whisper-large-v3"
!echo "Model folder: /content/drive/MyDrive/ai_models/whisper_large_v3"

In [ ]:
def ensure_model():

    print("\n🔍 Checking model in Drive...")

    if os.path.exists(LOCAL_DIR) and len(os.listdir(LOCAL_DIR)) > 0:
        print("✔ Model already exists in Drive — skipping download")
        return

    print("⬇ Model not found — downloading from Hugging Face...")

    os.makedirs(LOCAL_DIR, exist_ok=True)

    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=LOCAL_DIR,
        local_dir_use_symlinks=False
    )

    print("✔ Model downloaded and saved to Drive")

In [ ]:
def load_model():

    print("\n🚀 Loading Whisper model...")

    if torch.cuda.is_available():
        device = 0
        dtype = torch.float16
        print("🔥 GPU detected (FP16 enabled)")
    else:
        device = -1
        dtype = torch.float32
        print("⚠ CPU mode (slow)")

    pipe = pipeline(
        "automatic-speech-recognition",
        model=LOCAL_DIR,
        device=device,
        torch_dtype=dtype,
        model_kwargs={"attn_implementation": "sdpa"}
    )

    print("✔ Model loaded successfully")
    return pipe

In [ ]:
def split_audio_with_overlap(audio_path, chunk_ms=30000, overlap_ms=2000):

    print("\n✂ Splitting audio into chunks...")

    audio = AudioSegment.from_file(audio_path)

    chunks = []
    start = 0
    idx = 1

    while start < len(audio):

        end = start + chunk_ms
        chunk = audio[start:end]

        chunk_path = f"/content/chunk_{idx}.wav"
        chunk.export(chunk_path, format="wav")

        print(f"✔ Chunk {idx} created: {chunk_path}")

        chunks.append(chunk_path)

        start += (chunk_ms - overlap_ms)
        idx += 1

    print(f"📦 Total chunks created: {len(chunks)}")

    return chunks

In [ ]:
def clean_merge(text_list):

    print("\n🧩 Merging transcripts...")

    final_text = ""

    for i, t in enumerate(text_list):

        t = t.strip()

        if final_text and t[:30] in final_text[-200:]:
            t = t.split(" ", 3)[-1] if len(t.split()) > 3 else t

        print(f"✔ Merging chunk {i+1}")

        final_text += " " + t

    print("✔ Merge completed")

    return final_text.strip()

In [ ]:
def transcribe_long_audio(pipe, audio_path):

    print("\n🎧 Starting transcription pipeline...")

    chunks = split_audio_with_overlap(audio_path)

    results = []

    for i, chunk in enumerate(chunks):

        print(f"\n🧠 Transcribing chunk {i+1}/{len(chunks)}")

        out = pipe(
            chunk,
            chunk_length_s=30,
            batch_size=8,
            return_timestamps=False,
            generate_kwargs={
                "language": "hi",
                "task": "transcribe"
            }
        )

        print("✔ Chunk transcribed")

        results.append(out["text"])

    return clean_merge(results)

In [ ]:
!echo "🚀 Starting Whisper Pipeline..."

ensure_model()

pipe = load_model()

if not os.path.exists(AUDIO_FILE):
    print("❌ Audio file not found:", AUDIO_FILE)
    print("👉 Upload file to /content/audio.wav")
else:

    text = transcribe_long_audio(pipe, AUDIO_FILE)

    print("\n================ FINAL TRANSCRIPT ================\n")
    print(text)

    with open("/content/output.txt", "w", encoding="utf-8") as f:
        f.write(text)

    print("\n💾 Saved to /content/output.txt")
    print("🎉 Done successfully!")

In [ ]:
#to remove the chunk files
import os
import glob

print("🧹 Cleaning up chunk files and outputs...")

# Remove all chunk files
chunk_files = glob.glob("/content/chunk_*.wav")

for f in chunk_files:
    os.remove(f)
    print("🗑 Deleted:", f)

# Remove output text file
output_file = "/content/output.txt"

if os.path.exists(output_file):
    os.remove(output_file)
    print("🗑 Deleted: output.txt")

print("✔ Cleanup completed")